# 03 – Model Training & Evaluation
**Project:** SmartBank Customer Churn  
**Objective:** Train, evaluate, and select the best classification model. Propose evaluation metrics and business strategies.


In [1]:
import sys, warnings, pickle
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    roc_curve, precision_recall_curve, roc_auc_score,
    average_precision_score, classification_report,
    ConfusionMatrixDisplay, confusion_matrix
)

from src.data.load_data import load_raw_data
from src.data.preprocess import preprocess
from src.features.build_features import build_features
from src.models.train_model import train

sns.set_theme(style='whitegrid', palette='Set2')
print('Ready ✓')

Ready ✓


## 1. Prepare Data

In [2]:
raw   = load_raw_data()
clean = preprocess(raw, save=True)
X_train, X_test, y_train, y_test, scaler, encoder = build_features(clean)

print(f'X_train: {X_train.shape} | X_test: {X_test.shape}')
print(f'Train churn rate: {y_train.mean()*100:.1f}% | Test churn rate: {y_test.mean()*100:.1f}%')

2026-04-29 12:53:03 | INFO     | src.data.load_data | Loading sheet 'Customer_Demographics' …
2026-04-29 12:53:03 | INFO     | src.data.load_data |   → 1000 rows, 5 cols
2026-04-29 12:53:03 | INFO     | src.data.load_data | Loading sheet 'Transaction_History' …
2026-04-29 12:53:03 | INFO     | src.data.load_data |   → 5054 rows, 5 cols
2026-04-29 12:53:03 | INFO     | src.data.load_data | Loading sheet 'Customer_Service' …
2026-04-29 12:53:03 | INFO     | src.data.load_data |   → 1002 rows, 5 cols
2026-04-29 12:53:03 | INFO     | src.data.load_data | Loading sheet 'Online_Activity' …
2026-04-29 12:53:03 | INFO     | src.data.load_data |   → 1000 rows, 4 cols
2026-04-29 12:53:03 | INFO     | src.data.load_data | Loading sheet 'Churn_Status' …
2026-04-29 12:53:03 | INFO     | src.data.load_data |   → 1000 rows, 2 cols
2026-04-29 12:53:03 | INFO     | src.data.load_data | Merging all sources on CustomerID …
2026-04-29 12:53:03 | INFO     | src.data.load_data | Merge complete → 1000 rows, 

## 2. Train Models (Random Forest + XGBoost)

In [3]:
best_model, results = train(X_train, X_test, y_train, y_test)
print(f'Best model: {type(best_model).__name__}')

2026-04-29 12:53:07 | INFO     | src.models.train_model | Training Random Forest …
2026-04-29 12:53:10 | INFO     | src.models.train_model | CV ROC-AUC: 0.5888 ± 0.0400
2026-04-29 12:53:10 | INFO     | src.models.train_model | === Random Forest Evaluation ===
2026-04-29 12:53:10 | INFO     | src.models.train_model | ROC-AUC : 0.5243
2026-04-29 12:53:10 | INFO     | src.models.train_model | PR-AUC  : 0.2422
2026-04-29 12:53:10 | INFO     | src.models.train_model | 
              precision    recall  f1-score   support

    Retained       0.80      0.97      0.88       159
     Churned       0.29      0.05      0.08        41

    accuracy                           0.78       200
   macro avg       0.54      0.51      0.48       200
weighted avg       0.69      0.78      0.71       200

2026-04-29 12:53:10 | INFO     | src.models.train_model | Confusion matrix saved → C:\Users\amith\Downloads\customer-churn-project\customer-churn-project\models\confusion_matrix_random_forest.png
2026-04-

## 3. ROC Curves

In [4]:
fig, ax = plt.subplots(figsize=(8, 6))
colors = {'rf': 'steelblue', 'xgb': 'darkorange'}
labels = {'rf': 'Random Forest', 'xgb': 'XGBoost'}

for key, res in results.items():
    fpr, tpr, _ = roc_curve(y_test, res['y_proba'])
    ax.plot(fpr, tpr, lw=2,
            label=f"{labels.get(key, key)} (AUC={res['roc_auc']:.3f})",
            color=colors.get(key, 'grey'))

ax.plot([0,1],[0,1],'k--', lw=1, label='Random Baseline')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

## 4. Precision–Recall Curves

In [5]:
fig, ax = plt.subplots(figsize=(8, 6))

for key, res in results.items():
    prec, rec, _ = precision_recall_curve(y_test, res['y_proba'])
    ax.plot(rec, prec, lw=2,
            label=f"{labels.get(key, key)} (PR-AUC={res['pr_auc']:.3f})",
            color=colors.get(key, 'grey'))

baseline = y_test.mean()
ax.axhline(baseline, linestyle='--', color='black', label=f'Baseline (churn rate={baseline:.2f})')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision–Recall Curves', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Confusion Matrices

In [6]:
fig, axes = plt.subplots(1, len(results), figsize=(7*len(results), 5))
if len(results) == 1:
    axes = [axes]

for ax, (key, res) in zip(axes, results.items()):
    disp = ConfusionMatrixDisplay(res['conf_matrix'], display_labels=['Retained', 'Churned'])
    disp.plot(ax=ax, colorbar=False)
    ax.set_title(labels.get(key, key))

plt.suptitle('Confusion Matrices', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Classification Reports

In [7]:
for key, res in results.items():
    print(f'\n=== {labels.get(key, key)} ===')
    rpt = res['report']
    rpt_df = pd.DataFrame(rpt).T
    display(rpt_df.style.background_gradient(cmap='RdYlGn', vmin=0, vmax=1,
                                              subset=['precision','recall','f1-score']))


=== Random Forest ===


,precision,recall,f1-score,support
0,0.797927,0.968553,0.875000,159.000000
1,0.285714,0.048780,0.083333,41.000000
accuracy,0.780000,0.780000,0.780000,0.780000
macro avg,0.541821,0.508667,0.479167,200.000000
weighted avg,0.692924,0.780000,0.712708,200.000000



=== XGBoost ===


,precision,recall,f1-score,support
0,0.801205,0.836478,0.818462,159.000000
1,0.235294,0.195122,0.213333,41.000000
accuracy,0.705000,0.705000,0.705000,0.705000
macro avg,0.518249,0.515800,0.515897,200.000000
weighted avg,0.685193,0.705000,0.694410,200.000000


## 7. Feature Importances (Best Model)

In [8]:
if hasattr(best_model, 'feature_importances_'):
    importances = best_model.feature_importances_
    idx = np.argsort(importances)[::-1][:20]
    feat_names = list(X_train.columns)

    fig, ax = plt.subplots(figsize=(10, 7))
    ax.barh([feat_names[i] for i in reversed(idx)],
             importances[list(reversed(idx))], color='steelblue')
    ax.set_xlabel('Feature Importance')
    ax.set_title(f'Top-20 Feature Importances ({type(best_model).__name__})',
                  fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 8. Threshold Tuning

In [9]:
from sklearn.metrics import f1_score, precision_score, recall_score

best_key = max(results, key=lambda k: results[k]['roc_auc'])
y_proba  = results[best_key]['y_proba']

thresholds = np.arange(0.1, 0.9, 0.05)
f1s = [f1_score(y_test, (y_proba >= t).astype(int)) for t in thresholds]
precs = [precision_score(y_test, (y_proba >= t).astype(int)) for t in thresholds]
recs  = [recall_score(y_test, (y_proba >= t).astype(int)) for t in thresholds]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(thresholds, f1s,   label='F1',        lw=2, color='steelblue')
ax.plot(thresholds, precs, label='Precision',  lw=2, color='darkorange')
ax.plot(thresholds, recs,  label='Recall',     lw=2, color='green')
ax.axvline(thresholds[np.argmax(f1s)], ls='--', color='red',
            label=f'Best F1 threshold={thresholds[np.argmax(f1s)]:.2f}')
ax.set_xlabel('Decision Threshold')
ax.set_title('Threshold vs Metrics', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

best_t = thresholds[np.argmax(f1s)]
print(f'Optimal threshold (max F1): {best_t:.2f}')

Optimal threshold (max F1): 0.20


## 9. Model Evaluation Summary

### Chosen Metrics

| Metric | Rationale |
|---|---|
| **ROC-AUC** | Threshold-independent; measures overall discriminative power |
| **PR-AUC** | Focuses on the minority (churned) class — key for imbalanced datasets |
| **Recall (Churned)** | Priority: we must not miss actual churners (cost of losing a customer > cost of unnecessary outreach) |
| **Precision (Churned)** | Avoids flooding the retention team with false positives |
| **F1-Score** | Balanced view for operational decision-making |

### Business Recommendations

1. **Deployment:** Integrate the model into SmartBank's CRM; score all customers monthly.
2. **Threshold policy:** Use a lower threshold (0.35–0.40) to maximise recall during campaign season.
3. **SHAP explainability:** Add SHAP values in v2 so relationship managers can see *why* a customer is flagged.
4. **Retraining cadence:** Retrain quarterly with new transaction data to prevent model drift.
5. **A/B testing:** Measure retention rate uplift by comparing intervention vs control groups.
